In [1]:
from run_vlm_eval import main, load_config, set_envs, log_first_batch

/pasteur/u/rdcunha/code/mmbu/inference/.venv/lib/python3.13/site-packages/torch/cuda/__init__.py:63: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


INFO 01-09 11:01:23 [__init__.py:216] Automatically detected platform cuda.


In [2]:
import os
import json
import yaml
import torch
import pandas as pd
from tqdm import tqdm
from torch.utils.data import DataLoader

from vqa_dataset import PromptDataset, prompt_collate, create_template
from models import load_model_adapter

In [3]:
cfg = load_config("../configs/test_config.yaml")
model_cfg = cfg["model"]
tasks_cfg = cfg["tasks"]
run_cfg  = cfg["runtime"]
output_dir = '/pasteur/u/rdcunha/code/mmbu/results'

model_type = model_cfg["type"]
model_name = model_cfg["name"]
device     = model_cfg.get("device", "auto")
cache_dir  = "/pasteur/u/rdcunha/models"

set_envs(cache_dir)

In [4]:
adapter = load_model_adapter(model_type, model_name, device, cache_dir)
model, processor = adapter.load()

os.makedirs(output_dir, exist_ok=True)
file_model_name = model_name.split('/')[-1]
model_path = file_model_name.replace('/', '_')
output_dir = os.path.join(output_dir, model_path)
os.makedirs(output_dir, exist_ok=True)

/pasteur/u/rdcunha/code/mmbu/inference/.venv/lib/python3.13/site-packages/transformers/models/auto/modeling_auto.py:2284: FutureWarning: The class `AutoModelForVision2Seq` is deprecated and will be removed in v5.0. Please use `AutoModelForImageTextToText` instead.
  warnings.warn(
`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

In [ ]:
base_path = '/pasteur/u/rdcunha/data_cache/mmbu/final_data/subsampled_mmbu_data'

for task_cfg in tasks_cfg:
    print(f"Running task: {task_cfg['name']}")
    out_file = os.path.join(output_dir, f"{file_model_name.replace('/', '_')}_{task_cfg['name']}.jsonl")
    tsv_path = os.path.join(base_path, task_cfg["data_path"])
    df = pd.read_csv(tsv_path, sep='\t')
    
    add_options = ("open" not in task_cfg["name"])
    dataset = PromptDataset(df=df, add_options=add_options)
    loader = DataLoader(
        dataset,
        batch_size=run_cfg["batch_size"],
        shuffle=False,
        collate_fn=prompt_collate,
        num_workers=4,
        persistent_workers=True,
        pin_memory=True,
        prefetch_factor=4
    )

    existing = set()
    if os.path.exists(out_file):
        with open(out_file, "r") as f:
            for line in f:
                try:
                    j = json.loads(line)
                    existing.add(j["index"])
                except:
                    pass
                    
    counter = 0
    saved = []
    first_batch_logged = False
    
    with open(out_file, "a") as f:
        for batch in tqdm(loader, desc="Inference"):
    
            new_batch = [x for x in batch if x["index"] not in existing]
            if not new_batch:
                continue
    
            # inference
            # try:
                # messages = [create_template(item) for item in new_batch]
            messages = [adapter.create_template(item) for item in new_batch]
            # model-specific input prep
            inputs = adapter.prepare_inputs(messages, processor, model)
            outputs = adapter.infer(model, processor, inputs, run_cfg["max_new_tokens"])
            # except: 
            #     print(f"could not generate for {batch}")
            #     continue
    
            # log first batch only
            if run_cfg["log_first_batch"] and not first_batch_logged:
                log_first_batch(outputs, output_dir)
                first_batch_logged = True
    
            # save results
            for it, out_text in zip(new_batch, outputs):
                obj = {
                    "index": it["index"],
                    "question": it["question"],
                    "image_path": it["image_path"],
                    "dataset": it["dataset"],
                    "modality": it["modality"],
                    "class_label": it["class_label"],
                    "answer": out_text
                }
                if "options" in it and it["options"] is not None:
                    obj["options"] = it["options"]
            
                saved.append(obj)
                existing.add(it["index"])
                counter += 1
    
                if counter % 50 == 0:
                    for s in saved:
                        f.write(json.dumps(s) + "\n")
                    f.flush()
                    saved = []
    
        # Save remainder
        for s in saved:
            f.write(json.dumps(s) + "\n")

print('Completed')

Running task: detection_guess_bbox_closed_VQA


Inference:   0%|                         | 0/323 [00:00<?, ?it/s]The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.
Inference:   1%|               | 2/323 [00:44<1:58:05, 22.07s/it]A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.


In [5]:
from PIL import Image, ImageFile
import cv2

path = "/pasteur/u/rdcunha/data_cache/mmbu/final_data/VLMEvalData_v2/LMUData/standarized-subsampled/extra_det_v2/malaria-bounding-boxes/det/images_with_bbox/8757be1e-b832-407a-8e95-62abae485b24__bbox.png"

print("Testing PIL:")
try:
    ImageFile.LOAD_TRUNCATED_IMAGES = True
    img = Image.open(path)
    img.load()
    print("PIL loaded successfully")
except Exception as e:
    print("PIL error:", e)

print("\nTesting OpenCV:")
img_cv = cv2.imread(path)
print("OpenCV loaded:", img_cv is not None)

Testing PIL:
PIL loaded successfully

Testing OpenCV:
OpenCV loaded: False


libpng error: IDAT: CRC error


# Delete any results

In [4]:
from pathlib import Path

def delete_segmentation_jsonl(
    root="/pasteur/u/rdcunha/code/mmbu/results_cot_v3",
    substring="segmentation",
    dry_run=True,
    exclude_substrings=("extra", "none", "questions"),
):
    root = Path(root)
    if not root.exists():
        raise FileNotFoundError(f"Root path does not exist: {root}")

    sub = substring.lower()
    excludes = tuple(s.lower() for s in (exclude_substrings or ()) if s)

    targets = []
    for p in root.rglob("*.jsonl"):
        name = p.name.lower()
        if sub in name and not any(excl in name for excl in excludes):
            targets.append(p)

    print(f"Found {len(targets)} matching .jsonl files under {root}")
    for p in targets[:20]:
        print("  ", p)
    if len(targets) > 20:
        print(f"  ... and {len(targets) - 20} more")

    if dry_run:
        print("\nDry run: no files deleted. Set dry_run=False to delete.")
        return targets

    deleted = 0
    for p in targets:
        try:
            p.unlink()
            deleted += 1
        except Exception as e:
            print(f"Failed to delete {p}: {type(e).__name__}: {e}")

    print(f"\nDeleted {deleted}/{len(targets)} files.")
    return targets

# 1) Dry run first
# _ = delete_segmentation_jsonl(substring="detection_grounding", dry_run=True)
# _ = delete_segmentation_jsonl(substring='classification_questions', dry_run=True)

# 2) Then actually delete
_ = delete_segmentation_jsonl(substring="detection_grounding", dry_run=False)
# _ = delete_segmentation_jsonl(substring='classification_questions', dry_run=False)

Found 34 matching .jsonl files under /pasteur/u/rdcunha/code/mmbu/results_cot_v3
   /pasteur/u/rdcunha/code/mmbu/results_cot_v3/Qwen3-VL-4B-Instruct/Qwen3-VL-4B-Instruct_detection_grounding_closed_VQA_cot.jsonl
   /pasteur/u/rdcunha/code/mmbu/results_cot_v3/Qwen3-VL-4B-Instruct/Qwen3-VL-4B-Instruct_detection_grounding_open_VQA_cot.jsonl
   /pasteur/u/rdcunha/code/mmbu/results_cot_v3/Qwen3-VL-32B-Instruct/Qwen3-VL-32B-Instruct_detection_grounding_closed_VQA_cot.jsonl
   /pasteur/u/rdcunha/code/mmbu/results_cot_v3/Qwen3-VL-32B-Instruct/Qwen3-VL-32B-Instruct_detection_grounding_open_VQA_cot.jsonl
   /pasteur/u/rdcunha/code/mmbu/results_cot_v3/Qwen2.5-VL-32B-Instruct/Qwen2.5-VL-32B-Instruct_detection_grounding_open_VQA_cot.jsonl
   /pasteur/u/rdcunha/code/mmbu/results_cot_v3/Qwen2.5-VL-32B-Instruct/Qwen2.5-VL-32B-Instruct_detection_grounding_closed_VQA_cot.jsonl
   /pasteur/u/rdcunha/code/mmbu/results_cot_v3/InternVL3_5-8B/InternVL3_5-8B_detection_grounding_open_VQA_cot.jsonl
   /pasteur/u

# remove specific indices from results

In [5]:
from pathlib import Path

def truncate_jsonl_rows(
    root="/pasteur/u/rdcunha/code/mmbu/results_cot",
    substring="segmentation",
    cutoff_index=11600, # The last index you want to KEEP
    dry_run=True,
    exclude_substrings=("extra", "questions"),
):
    """
    Finds matching .jsonl files and removes all rows with an index > cutoff_index.
    (Keeps indices 0 through cutoff_index).
    """
    root = Path(root)
    if not root.exists():
        raise FileNotFoundError(f"Root path does not exist: {root}")

    sub = substring.lower()
    excludes = tuple(s.lower() for s in (exclude_substrings or ()) if s)

    # 1. Identify target files
    targets = []
    for p in root.rglob("*.jsonl"):
        name = p.name.lower()
        if sub in name and not any(excl in name for excl in excludes):
            targets.append(p)

    print(f"Found {len(targets)} matching .jsonl files under {root}")
    if not targets:
        return []

    # 2. Process files
    files_modified = 0
    total_lines_removed = 0
    
    # We want to keep indices 0..cutoff_index, so we keep (cutoff_index + 1) lines.
    lines_to_keep_count = cutoff_index + 1

    for p in targets:
        try:
            with open(p, 'r', encoding='utf-8') as f:
                lines = f.readlines()
            
            current_len = len(lines)
            
            # If file has fewer or equal lines than what we want to keep, skip it
            if current_len <= lines_to_keep_count:
                if dry_run:
                    print(f"  [SKIP] {p.name} (Max Index: {current_len-1} <= {cutoff_index})")
                continue

            # Slice to keep 0..cutoff_index
            kept_lines = lines[:lines_to_keep_count]
            lines_removed_count = current_len - len(kept_lines)
            
            if dry_run:
                print(f"  [WOULD TRUNCATE] {p.name}")
                print(f"     Current lines: {current_len} (Indices 0..{current_len-1})")
                print(f"     New lines:     {len(kept_lines)} (Indices 0..{cutoff_index})")
                print(f"     Removing:      {lines_removed_count} lines (Indices {cutoff_index+1}..{current_len-1})")
            else:
                with open(p, 'w', encoding='utf-8') as f:
                    f.writelines(kept_lines)
                print(f"  [TRUNCATED] {p.name}: Kept indices 0..{cutoff_index}")
                files_modified += 1
                total_lines_removed += lines_removed_count

        except Exception as e:
            print(f"Failed to process {p}: {type(e).__name__}: {e}")

    # 3. Summary
    if dry_run:
        print("\nDry run complete. No files were modified.")
        print(f"Set dry_run=False to delete indices > {cutoff_index}.")
    else:
        print(f"\nOperation complete. Modified {files_modified}/{len(targets)} files.")
        print(f"Total lines removed: {total_lines_removed}")

    return targets

# --- Usage ---
# truncate_jsonl_rows(substring="classification", cutoff_index=11600, dry_run=True)

# REAL RUN
truncate_jsonl_rows(substring="classification", cutoff_index=11600, dry_run=False)

Found 38 matching .jsonl files under /pasteur/u/rdcunha/code/mmbu/results_cot
  [TRUNCATED] medgemma-4b-it_classification_open_VQA_cot.jsonl: Kept indices 0..11600
  [TRUNCATED] medgemma-4b-it_classification_closed_VQA_cot.jsonl: Kept indices 0..11600
  [TRUNCATED] gemma-3-4b-it_classification_open_VQA_cot.jsonl: Kept indices 0..11600
  [TRUNCATED] gemma-3-4b-it_classification_closed_VQA_cot.jsonl: Kept indices 0..11600
  [TRUNCATED] Qwen3-VL-4B-Instruct_classification_closed_VQA_cot.jsonl: Kept indices 0..11600
  [TRUNCATED] Qwen3-VL-4B-Instruct_classification_open_VQA_cot.jsonl: Kept indices 0..11600
  [TRUNCATED] Qwen3-VL-8B-Instruct_classification_open_VQA_cot.jsonl: Kept indices 0..11600
  [TRUNCATED] Qwen3-VL-8B-Instruct_classification_closed_VQA_cot.jsonl: Kept indices 0..11600
  [TRUNCATED] Qwen2.5-VL-7B-Instruct_classification_closed_VQA_cot.jsonl: Kept indices 0..11600
  [TRUNCATED] Qwen2.5-VL-7B-Instruct_classification_open_VQA_cot.jsonl: Kept indices 0..11600
  [TRUNCATED] 

[PosixPath('/pasteur/u/rdcunha/code/mmbu/results_cot/medgemma-4b-it/medgemma-4b-it_classification_open_VQA_cot.jsonl'),
 PosixPath('/pasteur/u/rdcunha/code/mmbu/results_cot/medgemma-4b-it/medgemma-4b-it_classification_closed_VQA_cot.jsonl'),
 PosixPath('/pasteur/u/rdcunha/code/mmbu/results_cot/gemma-3-4b-it/gemma-3-4b-it_classification_open_VQA_cot.jsonl'),
 PosixPath('/pasteur/u/rdcunha/code/mmbu/results_cot/gemma-3-4b-it/gemma-3-4b-it_classification_closed_VQA_cot.jsonl'),
 PosixPath('/pasteur/u/rdcunha/code/mmbu/results_cot/Qwen3-VL-4B-Instruct/Qwen3-VL-4B-Instruct_classification_closed_VQA_cot.jsonl'),
 PosixPath('/pasteur/u/rdcunha/code/mmbu/results_cot/Qwen3-VL-4B-Instruct/Qwen3-VL-4B-Instruct_classification_open_VQA_cot.jsonl'),
 PosixPath('/pasteur/u/rdcunha/code/mmbu/results_cot/Qwen3-VL-8B-Instruct/Qwen3-VL-8B-Instruct_classification_open_VQA_cot.jsonl'),
 PosixPath('/pasteur/u/rdcunha/code/mmbu/results_cot/Qwen3-VL-8B-Instruct/Qwen3-VL-8B-Instruct_classification_closed_VQA_c

# move files between results and results_cot

In [2]:
import shutil
from pathlib import Path

def copy_questions_files(
    src_root="/pasteur/u/rdcunha/code/mmbu/results",
    dest_root="/pasteur/u/rdcunha/code/mmbu/results_cot",
    substring="_questions",
    dry_run=True
):
    """
    Copies files containing `substring` from src_root to dest_root,
    preserving the 'model name' subfolder structure.
    Skips copying if the destination file already exists.
    """
    src_path = Path(src_root)
    dest_path = Path(dest_root)

    if not src_path.exists():
        raise FileNotFoundError(f"Source root does not exist: {src_path}")

    # Create destination root if it doesn't exist
    if not dry_run:
        dest_path.mkdir(parents=True, exist_ok=True)

    # Find all matching files recursively
    targets = []
    for p in src_path.rglob("*"):
        if p.is_file() and substring in p.name:
            targets.append(p)

    print(f"Found {len(targets)} matching files containing '{substring}' in {src_root}")

    copied_count = 0
    skipped_count = 0

    for src_file in targets:
        # Determine relative path (e.g., "ModelName/file_questions.jsonl")
        rel_path = src_file.relative_to(src_path)
        
        # Construct full destination path
        dest_file = dest_path / rel_path
        
        # Check if destination already exists
        if dest_file.exists():
            skipped_count += 1
            continue

        if dry_run:
            print(f"  [WOULD COPY] {rel_path}")
        else:
            try:
                # Ensure the specific subfolder (model name) exists in destination
                dest_file.parent.mkdir(parents=True, exist_ok=True)
                
                # Copy the file
                shutil.copy2(src_file, dest_file)
                print(f"  [COPIED] {rel_path}")
                copied_count += 1
            except Exception as e:
                print(f"  [ERROR] Could not copy {rel_path}: {e}")

    # Summary
    if dry_run:
        print("\nDry run complete. No files moved.")
        print(f"Set dry_run=False to copy {len(targets) - skipped_count} files.")
    else:
        print(f"\nOperation complete.")
        print(f"Copied: {copied_count}")
        print(f"Skipped (already existed): {skipped_count}")

# --- Usage ---

# 1. Check what will happen
# copy_questions_files(dry_run=True)

# 2. Execute the copy
copy_questions_files(dry_run=False)

Found 71 matching files containing '_questions' in /pasteur/u/rdcunha/code/mmbu/results
  [COPIED] llava-med-v1.5-mistral-7b/llava-med-v1.5-mistral-7b_classification_questions.jsonl
  [COPIED] llava-med-v1.5-mistral-7b/llava-med-v1.5-mistral-7b_segmentation_questions.jsonl
  [COPIED] llava-med-v1.5-mistral-7b/llava-med-v1.5-mistral-7b_detection_questions.jsonl
  [COPIED] llava-med-v1.5-mistral-7b/llava-med-v1.5-mistral-7b_detection_grounding_questions.jsonl
  [COPIED] llava-1.5-7b-hf/llava-1.5-7b-hf_detection_questions.jsonl
  [COPIED] llava-1.5-7b-hf/llava-1.5-7b-hf_classification_questions.jsonl
  [COPIED] llava-1.5-7b-hf/llava-1.5-7b-hf_detection_grounding_questions.jsonl
  [COPIED] llava-1.5-7b-hf/llava-1.5-7b-hf_segmentation_questions.jsonl
  [COPIED] Qwen2-VL-2B-Instruct/Qwen2-VL-2B-Instruct_detection_questions.jsonl
  [COPIED] Qwen2-VL-2B-Instruct/Qwen2-VL-2B-Instruct_classification_questions.jsonl
  [COPIED] Qwen2-VL-2B-Instruct/Qwen2-VL-2B-Instruct_segmentation_questions.jsonl

In [14]:
import pandas as pd
import os

root = "/pasteur/u/rdcunha/data_cache/mmbu/final_data/subsampled_mmbu_data"
path = "final_cot_v2/det_guess_bbox_open.tsv"
tsv_path = os.path.join(root, path)
df = pd.read_csv(tsv_path, sep="\t")
len(df)

4238

In [15]:
df['index'].nunique()

4238